# LAB 10 — Extractive Question Answering (SQuAD)
**Môn:** Xử lý ngôn ngữ tự nhiên và ứng dụng — HK I 2026

**Cách dùng trên Google Colab:** `Runtime → Change runtime type → T4 GPU`, sau đó `Runtime → Run all`.
Tổng thời gian ước tính trên T4: khoảng 25–40 phút với cấu hình mặc định (có thể giảm `TRAIN_N`, `EXP1_EPOCHS` ở cell cấu hình nếu cần chạy nhanh).

**Nội dung notebook**
1. Cấu hình & reproducibility
2. Tokenization, offset mapping, sliding window (mục 10, 11, 13, 15)
3. Character span → token span (mục 16) + kiểm tra các trường hợp biên
4. Metric EM / F1 (mục 18, 19) + ví dụ tính tay F1
5. Baseline keyword-overlap (mục 21)
6. Hàm huấn luyện QA (start/end loss)
7. **Experiment 1** — context length
8. **Experiment 2** — fine-tune pretrained Transformer + learning curves
9. **Experiment 3** — EM vs F1
10. **Experiment 4** — question/context lexical overlap
11. **Experiment 5** — answer length
12. Wrong start vs wrong end + bảng error analysis
13. Document QA mini system (mục 27)
14. Reproducibility log, lưu `results.csv` và các hình
15. **Individual Learning Check** (12 câu hỏi, mục 33)

> **Lưu ý theo quy định AI Usage của lab:** các phần *prediction, giải thích kết quả, error analysis, reflection* phải do chính bạn viết.
> Trong notebook, những chỗ đó được để sẵn cell `✍️ TODO` — hãy điền sau khi đã nhìn thấy số liệu thật của bạn. Phần code (alignment, metric...) cần được bạn đọc hiểu, kiểm tra lại và ghi vào mục "AI-assisted code" ở cuối.

### ✍️ Prediction (ghi TRƯỚC khi chạy thí nghiệm — chép sang `prediction.md`)
- **P1.** Question và context có nhiều từ giống nhau nhưng chỉ một vị trí thật sự trả lời → model có dễ nhầm không?
- **P2.** Answer span càng dài thì boundary prediction có khó hơn không?
- **P3.** Context có nhiều thông tin không liên quan / distractor → performance thay đổi thế nào?
- **P4.** Đổi thứ tự một số câu trong context nhưng giữ nguyên answer → model có bị ảnh hưởng không?

*(Viết dự đoán của bạn ở đây:)*

## 1. Cài đặt & cấu hình

In [ ]:
!pip -q install -U transformers datasets accelerate

In [ ]:
import os, re, time, json, random, string, collections, platform
from collections import Counter
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForQuestionAnswering, get_linear_schedule_with_warmup

pd.set_option("display.max_colwidth", 90)
pd.set_option("display.width", 200)

# ----------------------------- CẤU HÌNH (đổi ở đây) -----------------------------
SEED          = 42
MODEL_NAME    = "distilbert-base-uncased"   # nhỏ, nhanh. Có thể thử "bert-base-uncased", "roberta-base"
TRAIN_N       = 10000     # số câu hỏi train (SQuAD train có ~87.6k)
VAL_N         = 1000      # validation: theo dõi learning curve
TEST_N        = 1500      # test: lấy từ phần còn lại của SQuAD validation (SQuAD không công khai test)
BATCH_SIZE    = 16
LR            = 3e-5
WEIGHT_DECAY  = 0.01
WARMUP_RATIO  = 0.1

# Core model (Experiment 2)
MAIN_MAX_LEN, MAIN_STRIDE, MAIN_EPOCHS = 384, 128, 3

# Experiment 1: giữ stride & epochs cố định, chỉ đổi max_length
EXP1_LENGTHS  = [128, 256, 384]
EXP1_STRIDE   = 32          # stride=128 không dùng được với max_length=128 (overlap >= độ dài window)
EXP1_EPOCHS   = 2

MAX_ANSWER_LEN = 30         # số token tối đa của span khi decode
N_BEST         = 20

device  = "cuda" if torch.cuda.is_available() else "cpu"
use_amp = device == "cuda"
print("device:", device, "|", torch.cuda.get_device_name(0) if use_amp else "CPU (sẽ rất chậm!)")

def set_seed(seed=SEED):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
set_seed()

### Dữ liệu: SQuAD v1.1

In [ ]:
raw = load_dataset("rajpurkar/squad")
print(raw)

train_raw = raw["train"].shuffle(seed=SEED).select(range(TRAIN_N))
val_full  = raw["validation"].shuffle(seed=SEED)
val_raw   = val_full.select(range(VAL_N))
test_raw  = val_full.select(range(VAL_N, VAL_N + TEST_N))
print(len(train_raw), len(val_raw), len(test_raw))

ex = train_raw[0]
print("\nQuestion :", ex["question"])
print("Answer   :", ex["answers"])
print("Context  :", ex["context"][:300], "...")

## 2. Tokenization của QA (mục 10, 11, 13, 15)
Input có dạng `[CLS] question [SEP] context [SEP]`. Với `return_overflowing_tokens=True` và `stride`, context dài được chia thành nhiều **window có overlap**.
Cần kiểm tra: `input_ids`, `attention_mask`, `offset_mapping`, `overflow_to_sample_mapping` — và xem model có dùng `token_type_ids` hay không.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print("model_input_names:", tokenizer.model_input_names)

# Chọn 1 ví dụ có context dài để thấy hiện tượng nhiều window
def n_tokens(e): return len(tokenizer(e["question"], e["context"])["input_ids"])
long_ex = next(e for e in train_raw if n_tokens(e) > 300)

enc = tokenizer(long_ex["question"], long_ex["context"],
                truncation="only_second", max_length=128, stride=32,
                return_overflowing_tokens=True, return_offsets_mapping=True)

print("keys                       :", list(enc.keys()))
print("'token_type_ids' có trong output?", "token_type_ids" in enc,
      "(nếu False => model này không dùng token type IDs)")
print("số window                  :", len(enc["input_ids"]))
print("overflow_to_sample_mapping :", enc["overflow_to_sample_mapping"], "(window -> index của sample gốc)")
for w in range(len(enc["input_ids"])):
    print(f"  window {w}: len(input_ids)={len(enc['input_ids'][w])}, sum(attention_mask)={sum(enc['attention_mask'][w])}")

toks = tokenizer.convert_ids_to_tokens(enc["input_ids"][0])
view = pd.DataFrame({"token": toks, "sequence_id": enc.sequence_ids(0), "offset": enc["offset_mapping"][0]})
print("\nWindow 0 (sequence_id: None=special, 0=question, 1=context):")
display(view.head(30))

In [ ]:
# Overlap giữa window 0 và window 1 (do stride)
def ctx_tokens(w):
    seq = enc.sequence_ids(w)
    return [(t, o) for t, o, s in zip(tokenizer.convert_ids_to_tokens(enc["input_ids"][w]), enc["offset_mapping"][w], seq) if s == 1]
c0, c1 = ctx_tokens(0), ctx_tokens(1)
print("window 0 kết thúc ở char", c0[-1][1][1], "| window 1 bắt đầu ở char", c1[0][1][0])
print("=> hai window chung nhau vùng context từ char", c1[0][1][0], "đến", c0[-1][1][1])

## 3. Answer span alignment: character span → token span (mục 16)
Với mỗi window ta cần: nếu answer nằm **hoàn toàn** trong phần context của window → gán `(start_token, end_token)`; nếu không → gán `(0, 0)` (vị trí `[CLS]`, nghĩa là "window này không chứa đáp án").

In [ ]:
def align_span(offsets, seq_ids, start_char, end_char):
    # Trả về (start_tok, end_tok) trong window, hoặc None nếu answer không nằm trọn trong window.
    ctx = [i for i, s in enumerate(seq_ids) if s == 1]
    if not ctx:
        return None
    ts, te = ctx[0], ctx[-1]
    # answer bị cắt hoặc nằm ngoài window
    if not (offsets[ts][0] <= start_char and offsets[te][1] >= end_char):
        return None
    start_tok = end_tok = None
    for t in range(ts, te + 1):
        s, e = offsets[t]
        if s <= start_char < e:      # token chứa ký tự đầu của answer (kể cả khi answer bắt đầu giữa token)
            start_tok = t
        if s < end_char <= e:        # token chứa ký tự cuối của answer (kể cả khi answer kết thúc giữa token)
            end_tok = t
    # fallback: ranh giới answer rơi vào khoảng trắng giữa các token
    if start_tok is None:
        start_tok = next((t for t in range(ts, te + 1) if offsets[t][0] >= start_char), None)
    if end_tok is None:
        end_tok = max((t for t in range(ts, te + 1) if offsets[t][1] <= end_char), default=None)
    if start_tok is None or end_tok is None or start_tok > end_tok:
        return None
    return start_tok, end_tok


def prepare_train_features(examples, max_length, stride):
    questions = [q.strip() for q in examples["question"]]
    tok = tokenizer(questions, examples["context"],
                    truncation="only_second", max_length=max_length, stride=stride,
                    return_overflowing_tokens=True, return_offsets_mapping=True,
                    padding="max_length")
    sample_map = tok.pop("overflow_to_sample_mapping")
    offset_map = tok.pop("offset_mapping")
    starts, ends, ex_ids = [], [], []
    for i, offsets in enumerate(offset_map):
        ans = examples["answers"][sample_map[i]]
        sc = ans["answer_start"][0]
        ec = sc + len(ans["text"][0])
        res = align_span(offsets, tok.sequence_ids(i), sc, ec)
        if res is None:
            starts.append(0); ends.append(0)         # [CLS]
        else:
            starts.append(res[0]); ends.append(res[1])
        ex_ids.append(examples["id"][sample_map[i]])
    tok["start_positions"] = starts
    tok["end_positions"] = ends
    tok["example_id"] = ex_ids
    return tok


def prepare_eval_features(examples, max_length, stride):
    questions = [q.strip() for q in examples["question"]]
    tok = tokenizer(questions, examples["context"],
                    truncation="only_second", max_length=max_length, stride=stride,
                    return_overflowing_tokens=True, return_offsets_mapping=True,
                    padding="max_length")
    sample_map = tok.pop("overflow_to_sample_mapping")
    tok["example_id"] = []
    for i in range(len(tok["input_ids"])):
        seq = tok.sequence_ids(i)
        tok["example_id"].append(examples["id"][sample_map[i]])
        # chỉ giữ offset của token thuộc context; còn lại = None để khi decode không chọn nhầm question/special token
        tok["offset_mapping"][i] = [o if seq[k] == 1 else None for k, o in enumerate(tok["offset_mapping"][i])]
    return tok

### Kiểm tra các trường hợp biên (mục 16)

In [ ]:
def show_alignment(question, context, answer, max_length=64, stride=16, title=""):
    s = context.index(answer); e = s + len(answer)
    enc = tokenizer(question, context, truncation="only_second", max_length=max_length, stride=stride,
                    return_overflowing_tokens=True, return_offsets_mapping=True)
    print(f"--- {title} | answer={answer!r} | char span=({s},{e}) | #windows={len(enc['input_ids'])}")
    results = []
    for w in range(len(enc["input_ids"])):
        r = align_span(enc["offset_mapping"][w], enc.sequence_ids(w), s, e)
        results.append(r)
        if r is None:
            print(f"   window {w}: answer không nằm trọn trong window  -> label = (0,0) [CLS]")
        else:
            toks = tokenizer.convert_ids_to_tokens(enc["input_ids"][w][r[0]:r[1]+1])
            off  = (enc["offset_mapping"][w][r[0]][0], enc["offset_mapping"][w][r[1]][1])
            print(f"   window {w}: start_tok={r[0]}, end_tok={r[1]}, tokens={toks}, char range của các token={off} -> {context[off[0]:off[1]]!r}")
    return results

q = "What is the capital of France?"
# 1) trường hợp thường
r1 = show_alignment(q, "The capital of France is Paris.", "Paris", title="Case 1: bình thường")
assert r1[0] is not None

# 2) answer bắt đầu / kết thúc GIỮA một token (sub-word)
q2 = "Which movement is mentioned?"
c2 = "The antidisestablishmentarianism movement was discussed."
show_alignment(q2, c2, "establishment", title="Case 2: answer bắt đầu & kết thúc giữa token (từ bị tách thành nhiều sub-word)")

# 3) answer bị truncation / nằm ngoài một số window
c3 = " ".join(["filler"] * 120) + " The answer is Paris ."
r3 = show_alignment(q, c3, "Paris", max_length=32, stride=8, title="Case 3: context dài, nhiều window")
assert any(r is None for r in r3) and any(r is not None for r in r3)

In [ ]:
# Kiểm tra toàn bộ train features: decode span token -> text và so với answer gốc
sample = train_raw.select(range(200))
feats_dbg = sample.map(lambda x: prepare_train_features(x, 384, 128), batched=True, remove_columns=sample.column_names)
id2ans = {e["id"]: e["answers"]["text"][0] for e in sample}

n_pos, n_cls, n_ok = 0, 0, 0
bad = []
for f in feats_dbg:
    if f["start_positions"] == 0:
        n_cls += 1; continue
    n_pos += 1
    decoded = tokenizer.decode(f["input_ids"][f["start_positions"]: f["end_positions"] + 1])
    gold = id2ans[f["example_id"]]
    if decoded.replace(" ", "").lower() == gold.replace(" ", "").lower():
        n_ok += 1
    else:
        bad.append((gold, decoded))
print(f"windows có answer: {n_pos} | windows không có answer (label CLS): {n_cls}")
print(f"decode khớp answer gốc: {n_ok}/{n_pos}")
print("Một số trường hợp không khớp tuyệt đối (thường do lowercase/accent/sub-word):", bad[:5])

## 4. Metric: Exact Match & token-level F1 (mục 18, 19)
- Normalization chuẩn của SQuAD: lowercase → bỏ dấu câu → bỏ mạo từ `a/an/the` → chuẩn hoá khoảng trắng.
- Một câu hỏi SQuAD validation có thể có nhiều gold answers → lấy **max** theo các gold.

In [ ]:
def normalize_answer(s):
    s = s.lower()
    s = "".join(ch for ch in s if ch not in set(string.punctuation))
    s = re.sub(r"\b(a|an|the)\b", " ", s)
    return " ".join(s.split())

def em_single(pred, gold):
    return float(normalize_answer(pred) == normalize_answer(gold))

def f1_single(pred, gold):
    p, g = normalize_answer(pred).split(), normalize_answer(gold).split()
    if len(p) == 0 or len(g) == 0:
        return float(p == g)
    n_same = sum((Counter(p) & Counter(g)).values())
    if n_same == 0:
        return 0.0
    prec, rec = n_same / len(p), n_same / len(g)
    return 2 * prec * rec / (prec + rec)

def best_over_golds(fn, pred, golds):
    return max(fn(pred, g) for g in golds)

# ---- Ví dụ tính tay vs code ----
gold, pred = "Paris France", "France"
print("Gold:", gold, "| Pred:", pred)
print("  #overlap=1, #pred=1, #gold=2  =>  P = 1/1 = 1.0, R = 1/2 = 0.5, F1 = 2*1*0.5/(1+0.5) = 0.6667")
print("  code: EM =", em_single(pred, gold), "| F1 =", round(f1_single(pred, gold), 4))

gold, pred = "Paris", "the city of Paris"
print("\nGold:", gold, "| Pred:", pred)
print("  sau normalize: pred = ['city','of','paris'] (bỏ 'the'), gold = ['paris'] -> P = 1/3, R = 1, F1 = 0.5, EM = 0")
print("  code: EM =", em_single(pred, gold), "| F1 =", round(f1_single(pred, gold), 4))

## 5. Baseline keyword-overlap (mục 21)
Heuristic đơn giản, **không dùng neural model**:
1. Tách context thành câu → chọn câu có nhiều content-word trùng với question nhất.
2. Trong câu đó, chấm điểm từng token "ứng viên" (không phải stop-word, không nằm trong question): ưu tiên token viết hoa / chứa số, ưu tiên gần các từ trùng với question.
3. Mở rộng sang các token liền kề cùng loại để tạo span.

In [ ]:
STOP = set("a an the of in on at to for and or is are was were be been being by with as from that this these those it its "
           "what which who whom whose when where why how do does did has have had not no can could will would".split())

def word_iter(text, base=0):
    return [(m.group(), base + m.start(), base + m.end()) for m in re.finditer(r"\w+", text)]

def sentence_spans(context):
    return [(m.start(), m.end()) for m in re.finditer(r"[^.!?]+[.!?]?", context) if m.group().strip()]

def baseline_predict(question, context):
    q_words = {w.lower() for w, _, _ in word_iter(question)} - STOP
    best_sent, best_score = None, -1
    for (ss, se) in sentence_spans(context):
        sw = {w.lower() for w, _, _ in word_iter(context[ss:se])}
        score = len(q_words & sw)
        if score > best_score:
            best_sent, best_score = (ss, se), score
    if best_sent is None:
        return {"text": "", "start_char": -1, "end_char": -1}
    ss, se = best_sent
    toks = word_iter(context[ss:se], base=ss)
    is_match = [t[0].lower() in q_words for t in toks]
    match_pos = [i for i, m in enumerate(is_match) if m]

    def entity_like(w):
        return w[0].isupper() or any(ch.isdigit() for ch in w)

    cand_scores = {}
    for i, (w, s, e) in enumerate(toks):
        if w.lower() in STOP or w.lower() in q_words:
            continue
        dist = min((abs(i - j) for j in match_pos), default=len(toks))
        cand_scores[i] = (1.0 if entity_like(w) else 0.0) - 0.05 * dist
    if not cand_scores:
        return {"text": context[ss:se].strip(), "start_char": ss, "end_char": se}
    b = max(cand_scores, key=cand_scores.get)
    kind = entity_like(toks[b][0])
    l = r = b
    while l - 1 in cand_scores and entity_like(toks[l - 1][0]) == kind and kind: l -= 1
    while r + 1 in cand_scores and entity_like(toks[r + 1][0]) == kind and kind: r += 1
    s, e = toks[l][1], toks[r][2]
    return {"text": context[s:e], "start_char": s, "end_char": e}

# thử nhanh
c = "The capital of France is Paris. The country is in Western Europe."
print(baseline_predict("What is the capital of France?", c))

## 6. Mô hình QA: start/end head + loss + decode
- `AutoModelForQuestionAnswering` = backbone + 1 lớp Linear(hidden → 2) → `start_logits`, `end_logits` (mỗi cái softmax **trên các vị trí token**).
- Loss: $L = \frac{L_{start}+L_{end}}{2}$, $L_{start} = -\log P_{start}(i)$, $L_{end} = -\log P_{end}(j)$ — đây chính là cách HuggingFace tính khi truyền `start_positions`, `end_positions`.
- Decode: chọn cặp $(i, j)$ với $i \le j$, $j - i + 1 \le$ `MAX_ANSWER_LEN`, điểm $= \text{start\_logit}_i + \text{end\_logit}_j$ cao nhất, **trên tất cả các window** của cùng một câu hỏi.

In [ ]:
def find_best_span(context, offs, sl, el, n_best=N_BEST, max_answer_len=MAX_ANSWER_LEN):
    best = None
    s_idx = np.argsort(sl)[-1:-n_best - 1:-1]
    e_idx = np.argsort(el)[-1:-n_best - 1:-1]
    for s in s_idx:
        for e in e_idx:
            if offs[s] is None or offs[e] is None:      # question / special token
                continue
            if e < s or e - s + 1 > max_answer_len:
                continue
            score = float(sl[s] + el[e])
            if best is None or score > best["score"]:
                best = {"text": context[offs[s][0]:offs[e][1]], "score": score,
                        "start_char": int(offs[s][0]), "end_char": int(offs[e][1]),
                        "start_tok": int(s), "end_tok": int(e)}
    return best


@torch.no_grad()
def get_logits(model, feats, batch_size=64):
    model.eval()
    cols = [c for c in tokenizer.model_input_names if c in feats.column_names]
    loader = DataLoader(feats.with_format("torch", columns=cols), batch_size=batch_size)
    S, E = [], []
    for batch in loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        with torch.autocast(device_type=device, dtype=torch.float16, enabled=use_amp):
            out = model(**batch)
        S.append(out.start_logits.float().cpu().numpy()); E.append(out.end_logits.float().cpu().numpy())
    return np.concatenate(S), np.concatenate(E)


def evaluate_model(model, raw_ds, max_length, stride):
    # Trả về EM, F1 (%), preds {id: text}, details {id: dict}
    feats = raw_ds.map(lambda x: prepare_eval_features(x, max_length, stride),
                       batched=True, remove_columns=raw_ds.column_names)
    S, E = get_logits(model, feats)
    feat_of = collections.defaultdict(list)
    for i, eid in enumerate(feats["example_id"]):
        feat_of[eid].append(i)
    offsets_all = feats["offset_mapping"]
    preds, details = {}, {}
    for ex in raw_ds:
        best = None
        for fi in feat_of[ex["id"]]:
            cand = find_best_span(ex["context"], offsets_all[fi], S[fi], E[fi])
            if cand is not None and (best is None or cand["score"] > best["score"]):
                best = cand
        details[ex["id"]] = best or {"text": "", "start_char": -1, "end_char": -1, "score": float("-inf")}
        preds[ex["id"]] = details[ex["id"]]["text"]
    em, f1 = score_predictions(raw_ds, preds)
    return em, f1, preds, details


def score_predictions(raw_ds, preds):
    ems, f1s = [], []
    for ex in raw_ds:
        golds = ex["answers"]["text"]
        ems.append(best_over_golds(em_single, preds[ex["id"]], golds))
        f1s.append(best_over_golds(f1_single, preds[ex["id"]], golds))
    return 100 * float(np.mean(ems)), 100 * float(np.mean(f1s))


def answer_coverage(raw_ds, max_length, stride):
    # tỉ lệ câu hỏi mà answer nằm trọn trong ÍT NHẤT một window (chặn trên của EM)
    f = raw_ds.map(lambda x: prepare_train_features(x, max_length, stride),
                   batched=True, remove_columns=raw_ds.column_names)
    df = pd.DataFrame({"id": f["example_id"], "has": [s != 0 for s in f["start_positions"]]})
    return 100 * df.groupby("id")["has"].any().mean(), len(f)

### Hàm huấn luyện

In [ ]:
def train_qa(max_length, stride, epochs, lr=LR, batch_size=BATCH_SIZE, seed=SEED, verbose=True):
    set_seed(seed)
    train_feats = train_raw.map(lambda x: prepare_train_features(x, max_length, stride),
                                batched=True, remove_columns=train_raw.column_names)
    val_feats   = val_raw.map(lambda x: prepare_train_features(x, max_length, stride),
                              batched=True, remove_columns=val_raw.column_names)
    cols = [c for c in tokenizer.model_input_names if c in train_feats.column_names] + ["start_positions", "end_positions"]
    train_loader = DataLoader(train_feats.with_format("torch", columns=cols), batch_size=batch_size, shuffle=True)
    val_loader   = DataLoader(val_feats.with_format("torch", columns=cols), batch_size=64)

    model = AutoModelForQuestionAnswering.from_pretrained(MODEL_NAME).to(device)
    no_decay = ["bias", "LayerNorm.weight"]
    params = [
        {"params": [p for n, p in model.named_parameters() if not any(nd in n for nd in no_decay)], "weight_decay": WEIGHT_DECAY},
        {"params": [p for n, p in model.named_parameters() if any(nd in n for nd in no_decay)], "weight_decay": 0.0},
    ]
    optimizer = torch.optim.AdamW(params, lr=lr)
    total_steps = epochs * len(train_loader)
    scheduler = get_linear_schedule_with_warmup(optimizer, int(WARMUP_RATIO * total_steps), total_steps)
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

    hist = {"step_loss": [], "epoch_end_step": [], "train_loss": [], "val_loss": [], "val_em": [], "val_f1": []}
    train_time = 0.0
    for ep in range(1, epochs + 1):
        model.train(); t0 = time.time(); run = []
        for batch in tqdm(train_loader, desc=f"[L={max_length}] epoch {ep}/{epochs}", disable=not verbose):
            batch = {k: v.to(device) for k, v in batch.items()}
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type=device, dtype=torch.float16, enabled=use_amp):
                loss = model(**batch).loss            # = (L_start + L_end) / 2
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer); scaler.update(); scheduler.step()
            hist["step_loss"].append(loss.item()); run.append(loss.item())
        if device == "cuda": torch.cuda.synchronize()
        train_time += time.time() - t0

        # validation loss
        model.eval(); tot, n = 0.0, 0
        with torch.no_grad():
            for batch in val_loader:
                batch = {k: v.to(device) for k, v in batch.items()}
                with torch.autocast(device_type=device, dtype=torch.float16, enabled=use_amp):
                    l = model(**batch).loss
                bs = batch["input_ids"].size(0); tot += l.item() * bs; n += bs
        em, f1, _, _ = evaluate_model(model, val_raw, max_length, stride)
        hist["epoch_end_step"].append(len(hist["step_loss"]))
        hist["train_loss"].append(float(np.mean(run))); hist["val_loss"].append(tot / n)
        hist["val_em"].append(em); hist["val_f1"].append(f1)
        if verbose:
            print(f"epoch {ep}: train_loss={hist['train_loss'][-1]:.4f} val_loss={hist['val_loss'][-1]:.4f} val_EM={em:.2f} val_F1={f1:.2f}")
    return {"model": model, "hist": hist, "train_time": train_time, "n_train_windows": len(train_feats)}

## 7. Experiment 1 — Context length (mục 14)
So sánh `max_length ∈ {128, 256, 384}`; giữ cố định: model, stride, epochs, lr, batch size, seed, dữ liệu.
Ngoài EM/F1/thời gian, ta đo thêm **answer coverage** (% câu test có answer nằm trọn trong ít nhất một window) — đây là chặn trên cho EM.

In [ ]:
exp1_rows = []
for L in EXP1_LENGTHS:
    out = train_qa(L, EXP1_STRIDE, EXP1_EPOCHS, verbose=True)
    em, f1, _, _ = evaluate_model(out["model"], test_raw, L, EXP1_STRIDE)
    cov, n_test_win = answer_coverage(test_raw, L, EXP1_STRIDE)
    exp1_rows.append({"Max length": L, "Stride": EXP1_STRIDE, "Epochs": EXP1_EPOCHS, "EM": round(em, 2), "F1": round(f1, 2),
                      "Training time (s)": round(out["train_time"], 1), "#train windows": out["n_train_windows"],
                      "Test answer coverage (%)": round(cov, 2)})
    del out; torch.cuda.empty_cache()
exp1_df = pd.DataFrame(exp1_rows)
display(exp1_df)

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(14, 3.8))
ax[0].plot(exp1_df["Max length"], exp1_df["EM"], "o-", label="EM"); ax[0].plot(exp1_df["Max length"], exp1_df["F1"], "s-", label="F1")
ax[0].set_title("Test EM / F1 vs max_length"); ax[0].set_xlabel("max_length"); ax[0].legend(); ax[0].grid(alpha=.3)
ax[1].bar(exp1_df["Max length"].astype(str), exp1_df["Training time (s)"]); ax[1].set_title("Training time (s)"); ax[1].set_xlabel("max_length")
ax[2].bar(exp1_df["Max length"].astype(str), exp1_df["Test answer coverage (%)"]); ax[2].set_title("Answer coverage trên test (%)"); ax[2].set_xlabel("max_length")
ax[2].set_ylim(max(0, exp1_df["Test answer coverage (%)"].min() - 5), 100.5)
plt.tight_layout(); plt.savefig("exp1_context_length.png", dpi=150); plt.show()

### ✍️ TODO — Phân tích Experiment 1
*Câu hỏi của lab:* **Tăng context length có luôn cải thiện QA không?**
Gợi ý để bạn tự viết: so sánh EM/F1 giữa các mức; xét cột *coverage* và *#train windows* (window nhỏ ⇒ nhiều window hơn, nhiều window nhãn `[CLS]` hơn); xét thời gian huấn luyện; chênh lệch có đủ lớn so với nhiễu do 1 seed không?

## 8. Experiment 2 — Fine-tune pretrained Transformer (mục 17)
Cấu hình core: `max_length=384`, `stride=128`. Báo cáo training loss, validation loss, EM, F1 và vẽ learning curves.

In [ ]:
main = train_qa(MAIN_MAX_LEN, MAIN_STRIDE, MAIN_EPOCHS)
main_model, main_hist, main_time = main["model"], main["hist"], main["train_time"]

In [ ]:
def smooth(x, k=25):
    x = np.array(x); 
    return np.convolve(x, np.ones(k) / k, mode="valid") if len(x) > k else x

fig, ax = plt.subplots(1, 3, figsize=(16, 4))
sl = main_hist["step_loss"]
ax[0].plot(sl, alpha=.25, label="train loss (step)")
ax[0].plot(np.arange(len(smooth(sl))) + 12, smooth(sl), label="train loss (moving avg)")
ax[0].scatter(main_hist["epoch_end_step"], main_hist["val_loss"], c="r", zorder=5, label="val loss (cuối epoch)")
ax[0].plot(main_hist["epoch_end_step"], main_hist["val_loss"], "r--")
ax[0].set_xlabel("step"); ax[0].set_ylabel("loss"); ax[0].set_title("Loss"); ax[0].legend(); ax[0].grid(alpha=.3)
eps = np.arange(1, len(main_hist["val_em"]) + 1)
ax[1].plot(eps, main_hist["train_loss"], "o-", label="train (mean/epoch)"); ax[1].plot(eps, main_hist["val_loss"], "s-", label="validation")
ax[1].set_xlabel("epoch"); ax[1].set_title("Train vs Val loss"); ax[1].legend(); ax[1].grid(alpha=.3); ax[1].set_xticks(eps)
ax[2].plot(eps, main_hist["val_em"], "o-", label="val EM"); ax[2].plot(eps, main_hist["val_f1"], "s-", label="val F1")
ax[2].set_xlabel("epoch"); ax[2].set_title("Validation EM / F1"); ax[2].legend(); ax[2].grid(alpha=.3); ax[2].set_xticks(eps)
plt.tight_layout(); plt.savefig("learning_curves.png", dpi=150); plt.show()

display(pd.DataFrame({"epoch": eps, "train_loss": main_hist["train_loss"], "val_loss": main_hist["val_loss"],
                      "val_EM": main_hist["val_em"], "val_F1": main_hist["val_f1"]}).round(4))
print(f"Training time: {main_time:.1f}s")

In [ ]:
# Đánh giá cuối cùng trên TEST
m_em, m_f1, m_preds, m_details = evaluate_model(main_model, test_raw, MAIN_MAX_LEN, MAIN_STRIDE)
print(f"Fine-tuned {MODEL_NAME}  |  TEST EM = {m_em:.2f}  F1 = {m_f1:.2f}")

### ✍️ TODO — Nhận xét learning curves
Mô hình có overfit không (so sánh train loss vs val loss)? EM/F1 còn tăng ở epoch cuối không?

## 9. Experiment 3 — EM vs F1 (mục 20): baseline vs fine-tuned Transformer

In [ ]:
b_details = {ex["id"]: baseline_predict(ex["question"], ex["context"]) for ex in test_raw}
b_preds = {k: v["text"] for k, v in b_details.items()}
b_em, b_f1 = score_predictions(test_raw, b_preds)

exp3_df = pd.DataFrame([{"Model": "Baseline (keyword overlap)", "EM": round(b_em, 2), "F1": round(b_f1, 2)},
                        {"Model": f"Fine-tuned {MODEL_NAME}",   "EM": round(m_em, 2), "F1": round(m_f1, 2)}])
display(exp3_df)

def per_example_df(raw_ds, details):
    rows = []
    for ex in raw_ds:
        d = details[ex["id"]]; golds = ex["answers"]["text"]; starts = ex["answers"]["answer_start"]
        f1s = [f1_single(d["text"], g) for g in golds]; k = int(np.argmax(f1s))   # gold khớp nhất
        rows.append({"id": ex["id"], "question": ex["question"], "context": ex["context"], "golds": golds,
                     "gold": golds[k], "gold_start": starts[k], "gold_end": starts[k] + len(golds[k]),
                     "pred": d["text"], "pred_start": d["start_char"], "pred_end": d["end_char"],
                     "EM": max(em_single(d["text"], g) for g in golds), "F1": max(f1s)})
    return pd.DataFrame(rows)

df_m = per_example_df(test_raw, m_details)
df_b = per_example_df(test_raw, b_details)
print("Fine-tuned: EM =", round(100 * df_m.EM.mean(), 2), "F1 =", round(100 * df_m.F1.mean(), 2),
      "| Baseline: EM =", round(100 * df_b.EM.mean(), 2), "F1 =", round(100 * df_b.F1.mean(), 2))

In [ ]:
# >= 5 ví dụ EM = 0 nhưng F1 > 0 (model fine-tuned)
partial = df_m[(df_m.EM == 0) & (df_m.F1 > 0)]
print(f"Có {len(partial)} / {len(df_m)} ví dụ EM=0 & F1>0 ({100*len(partial)/len(df_m):.1f}%)")
em0f1 = partial.sample(min(8, len(partial)), random_state=SEED)[["question", "gold", "pred", "EM", "F1"]].reset_index(drop=True)
em0f1["F1"] = em0f1["F1"].round(3)
em0f1["✍️ Giải thích (tự viết)"] = ""
display(em0f1)
em0f1.to_csv("em0_f1pos_examples.csv", index=False)

### ✍️ TODO — Giải thích vì sao EM = 0 nhưng F1 > 0
Với mỗi ví dụ ở bảng trên: prediction dài hơn hay ngắn hơn gold? Thiếu/thừa token nào? Có phải do normalization (mạo từ, dấu câu) hay do ranh giới span?

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
x = np.arange(2); w = 0.35
ax[0].bar(x - w/2, [b_em, b_f1], w, label="Baseline"); ax[0].bar(x + w/2, [m_em, m_f1], w, label="Fine-tuned")
ax[0].set_xticks(x); ax[0].set_xticklabels(["EM", "F1"]); ax[0].set_ylabel("%"); ax[0].set_title("Test: Baseline vs Fine-tuned"); ax[0].legend()
ax[1].hist(df_m.F1, bins=20); ax[1].set_title("Phân phối F1 theo từng ví dụ (fine-tuned)"); ax[1].set_xlabel("F1 per example")
plt.tight_layout(); plt.savefig("em_f1_analysis.png", dpi=150); plt.show()

## 10. Experiment 4 — Question/Context matching (mục 22)
Với mỗi câu test, đo **lexical overlap** giữa question và *câu chứa answer*:
$\text{overlap} = \frac{|\text{content words}(Q)\ \cap\ \text{words}(S_{answer})|}{|\text{content words}(Q)|}$.
Chia thành 3 nhóm (low / mid / high) theo tertile rồi so sánh EM/F1 của baseline và Transformer.
Nhóm `low` ≈ câu hỏi được diễn đạt khác với câu chứa đáp án (paraphrase).

In [ ]:
def answer_sentence(context, start_char):
    for (ss, se) in sentence_spans(context):
        if ss <= start_char < se:
            return context[ss:se]
    return context

def lexical_overlap(question, sentence):
    qw = {w.lower() for w, _, _ in word_iter(question)} - STOP
    sw = {w.lower() for w, _, _ in word_iter(sentence)}
    return len(qw & sw) / max(1, len(qw))

ov = df_m.copy()
ov["overlap"] = [lexical_overlap(q, answer_sentence(c, s)) for q, c, s in zip(ov.question, ov.context, ov.gold_start)]
ov["group"] = pd.qcut(ov["overlap"].rank(method="first"), 3, labels=["low", "mid", "high"])
ov["B_EM"], ov["B_F1"] = df_b.EM.values, df_b.F1.values

exp4 = ov.groupby("group", observed=True).agg(n=("id", "count"), mean_overlap=("overlap", "mean"),
                                              Transformer_EM=("EM", "mean"), Transformer_F1=("F1", "mean"),
                                              Baseline_EM=("B_EM", "mean"), Baseline_F1=("B_F1", "mean")).reset_index()
for c in ["Transformer_EM", "Transformer_F1", "Baseline_EM", "Baseline_F1"]:
    exp4[c] = (100 * exp4[c]).round(2)
exp4["mean_overlap"] = exp4["mean_overlap"].round(3)
display(exp4)

print("Ví dụ overlap THẤP (paraphrase):")
display(ov[ov.group == "low"].sample(4, random_state=SEED)[["question", "gold", "pred", "overlap", "F1"]].round(2))
print("Ví dụ overlap CAO:")
display(ov[ov.group == "high"].sample(4, random_state=SEED)[["question", "gold", "pred", "overlap", "F1"]].round(2))

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4)); x = np.arange(3); w = 0.2
ax.bar(x - 1.5*w, exp4.Transformer_EM, w, label="Transformer EM"); ax.bar(x - 0.5*w, exp4.Transformer_F1, w, label="Transformer F1")
ax.bar(x + 0.5*w, exp4.Baseline_EM, w, label="Baseline EM");       ax.bar(x + 1.5*w, exp4.Baseline_F1, w, label="Baseline F1")
ax.set_xticks(x); ax.set_xticklabels(exp4.group); ax.set_xlabel("question–answer-sentence lexical overlap"); ax.set_ylabel("%"); ax.legend(fontsize=8)
plt.tight_layout(); plt.savefig("exp4_overlap.png", dpi=150); plt.show()

### ✍️ TODO — Phân tích Experiment 4
Model có phụ thuộc mạnh vào lexical overlap không? Khoảng cách Transformer–baseline ở nhóm `low` so với `high` nói lên điều gì? *(đối chiếu với Prediction P1)*.

## 11. Experiment 5 — Answer length (mục 23)
Nhóm theo số từ (tách theo khoảng trắng) của gold answer: `1`, `2–3`, `4–6`, `7+`.

In [ ]:
ov["gold_len"] = ov.gold.str.split().str.len()
ov["len_group"] = pd.cut(ov.gold_len, bins=[0, 1, 3, 6, 10**6], labels=["1", "2-3", "4-6", "7+"])
exp5 = ov.groupby("len_group", observed=True).agg(n=("id", "count"), Transformer_EM=("EM", "mean"), Transformer_F1=("F1", "mean"),
                                                  Baseline_EM=("B_EM", "mean"), Baseline_F1=("B_F1", "mean")).reset_index()
for c in ["Transformer_EM", "Transformer_F1", "Baseline_EM", "Baseline_F1"]:
    exp5[c] = (100 * exp5[c]).round(2)
exp5 = exp5.rename(columns={"len_group": "Answer length"})
display(exp5)

fig, ax = plt.subplots(figsize=(7, 4)); x = np.arange(len(exp5)); w = 0.2
ax.bar(x - 1.5*w, exp5.Transformer_EM, w, label="Transformer EM"); ax.bar(x - 0.5*w, exp5.Transformer_F1, w, label="Transformer F1")
ax.bar(x + 0.5*w, exp5.Baseline_EM, w, label="Baseline EM");       ax.bar(x + 1.5*w, exp5.Baseline_F1, w, label="Baseline F1")
ax.set_xticks(x); ax.set_xticklabels(exp5["Answer length"]); ax.set_xlabel("độ dài gold answer (từ)"); ax.set_ylabel("%"); ax.legend(fontsize=8)
plt.tight_layout(); plt.savefig("answer_length_analysis.png", dpi=150); plt.show()

### ✍️ TODO — Phân tích Experiment 5
Câu trả lời dài có khó hơn không? Khoảng cách EM–F1 thay đổi thế nào theo độ dài? Đối chiếu với Prediction P2. (Chú ý cỡ mẫu `n` của nhóm `7+` nhỏ ⇒ kết luận cần thận trọng.)

## 12. Wrong start vs wrong end & Error analysis (mục 24, 25)
Với mỗi ví dụ **EM = 0**, so sánh vị trí ký tự của prediction với gold (chọn gold khớp nhất):
- `wrong_end`: start đúng, end sai
- `wrong_start`: end đúng, start sai
- `wrong_both`: cả hai sai → chia tiếp: `partial_overlap` (span giao nhau) hoặc `no_overlap` (sai hẳn vùng)

In [ ]:
err = df_m[df_m.EM == 0].copy()
def categorize(r):
    s_ok, e_ok = r.pred_start == r.gold_start, r.pred_end == r.gold_end
    if s_ok and not e_ok:  return "wrong_end"
    if e_ok and not s_ok:  return "wrong_start"
    overlap = max(0, min(r.pred_end, r.gold_end) - max(r.pred_start, r.gold_start))
    return "wrong_both_partial_overlap" if overlap > 0 else "wrong_both_no_overlap"
err["error_type_auto"] = err.apply(categorize, axis=1)

# hướng sai của ranh giới
def direction(r):
    if r.error_type_auto == "wrong_end":   return "pred ngắn hơn (end sớm)" if r.pred_end < r.gold_end else "pred dài hơn (end muộn)"
    if r.error_type_auto == "wrong_start": return "pred ngắn hơn (start muộn)" if r.pred_start > r.gold_start else "pred dài hơn (start sớm)"
    return ""
err["direction"] = err.apply(direction, axis=1)

counts = err.error_type_auto.value_counts()
print(f"Tổng EM=0: {len(err)} / {len(df_m)}")
display(counts.to_frame("count").assign(pct_of_errors=lambda d: (100 * d["count"] / len(err)).round(1)))
print("Hướng sai ranh giới:")
display(err[err.direction != ""].groupby(["error_type_auto", "direction"]).size().to_frame("count"))

n_wrong_start = int((err.error_type_auto == "wrong_start").sum())
n_wrong_end   = int((err.error_type_auto == "wrong_end").sum())
print(f"\n#wrong start = {n_wrong_start} | #wrong end = {n_wrong_end}")

In [ ]:
# Bảng ứng viên cho error_analysis.md: >= 5 mỗi loại (sample ngẫu nhiên, seed cố định)
tbl = []
for cat in ["wrong_start", "wrong_end", "wrong_both_partial_overlap", "wrong_both_no_overlap"]:
    sub = err[err.error_type_auto == cat]
    tbl.append(sub.sample(min(6, len(sub)), random_state=SEED))
cand = pd.concat(tbl)[["question", "gold", "pred", "error_type_auto", "F1"]].reset_index(drop=True)
cand["F1"] = cand["F1"].round(3)
cand["✍️ Error type (chọn: wrong start / wrong end / boundary / partial / distractor / insufficient context / long context / multiple plausible spans / ambiguity / truncation / tokenization)"] = ""
cand["✍️ Explanation"] = ""
display(cand)
cand.to_csv("error_analysis_candidates.csv", index=False)

In [ ]:
# Xem ngữ cảnh quanh gold & prediction của một ví dụ (đổi chỉ số i để xem ví dụ khác)
def inspect_example(i, win=150):
    r = err.reset_index(drop=True).iloc[i]
    print("Q   :", r.question); print("Gold:", r.gold, "| Pred:", r.pred, "|", r.error_type_auto)
    g0 = max(0, r.gold_start - win); print("Ngữ cảnh quanh gold:", "..." + r.context[g0: r.gold_end + win] + "...")
    if r.pred_start >= 0 and abs(r.pred_start - r.gold_start) > win:
        p0 = max(0, r.pred_start - win); print("Ngữ cảnh quanh pred:", "..." + r.context[p0: r.pred_end + win] + "...")
inspect_example(0)

### ✍️ TODO — Error analysis & giả thuyết
1. So sánh `#wrong start` và `#wrong end`; bảng `direction` cho thấy model hay cắt ngắn hay kéo dài span?
2. Đưa ra **giả thuyết** nguyên nhân (ví dụ: head start/end độc lập; ranh giới danh từ ghép; nhiều span hợp lệ; gold annotation không nhất quán...) và nêu cách kiểm chứng.
3. Điền cột *Error type* và *Explanation* cho ≥5 ví dụ mỗi loại trong `error_analysis_candidates.csv` → chép vào `error_analysis.md`.

## 13. Document QA — mini system (mục 27)
`User question → Context/document → QA model → Answer span`. Context được cung cấp trực tiếp (chưa có retrieval — đó là LAB 11).
Hàm `answer_question` dùng sliding window nên xử lý được tài liệu dài hơn 384 token.

In [ ]:
def answer_question(question, context, model=None, max_length=MAIN_MAX_LEN, stride=MAIN_STRIDE):
    model = model or main_model
    model.eval()
    tok = tokenizer(question.strip(), context, truncation="only_second", max_length=max_length, stride=stride,
                    return_overflowing_tokens=True, return_offsets_mapping=True, padding="max_length", return_tensors="pt")
    tok.pop("overflow_to_sample_mapping")
    offsets = tok.pop("offset_mapping").tolist()
    inputs = {k: v.to(device) for k, v in tok.items() if k in tokenizer.model_input_names}
    with torch.no_grad(), torch.autocast(device_type=device, dtype=torch.float16, enabled=use_amp):
        out = model(**inputs)
    S, E = out.start_logits.float().cpu().numpy(), out.end_logits.float().cpu().numpy()
    best = None
    for w in range(S.shape[0]):
        seq = tok.sequence_ids(w)
        offs = [o if seq[k] == 1 else None for k, o in enumerate(offsets[w])]
        cand = find_best_span(context, offs, S[w], E[w])
        if cand is not None and (best is None or cand["score"] > best["score"]):
            best = {**cand, "window": w}
    return best

document = (
    "Hanoi is the capital of Vietnam and its second-largest city by population. "
    "The city sits on the right bank of the Red River and has been the political centre of the country for many centuries. "
    "In 1010, Emperor Ly Thai To moved the royal capital to the site and named it Thang Long. "
    "The Temple of Literature, founded in 1070, is regarded as Vietnam's first national university. "
    "Today, the city is known for its Old Quarter, where the streets were historically named after the goods sold there. "
    "Hoan Kiem Lake lies in the centre of the city, and the Ngoc Son Temple stands on a small island in the lake. "
    "The Ho Chi Minh Mausoleum, completed in 1975, is located in Ba Dinh Square."
)
questions = [
    "What is the capital of Vietnam?",
    "Who moved the royal capital to Thang Long?",
    "When was the Temple of Literature founded?",
    "Where is the Ho Chi Minh Mausoleum located?",
    "Which river is the city located next to?",
    "What is the population of Hanoi?",          # KHÔNG có con số trong văn bản -> model vẫn trả về 1 span
]
rows = []
for q in questions:
    r = answer_question(q, document)
    rows.append({"question": q, "answer": r["text"] if r else "", "span(char)": (r["start_char"], r["end_char"]) if r else None,
                 "score(logit sum)": round(r["score"], 2) if r else None})
display(pd.DataFrame(rows))

In [ ]:
# (Tuỳ chọn) Thử câu hỏi / tài liệu của riêng bạn
my_context = document          # thay bằng một đoạn Wikipedia / FAQ / tài liệu học thuật của bạn
my_question = "Where does the Ngoc Son Temple stand?"
print(answer_question(my_question, my_context))

### ✍️ TODO — Nhận xét Document QA
Câu hỏi cuối (không có đáp án trong văn bản) cho kết quả gì? Điều đó nói gì về hạn chế của extractive QA khi **không** có cơ chế no-answer (SQuAD 2.0)?

## 14. Reproducibility & lưu kết quả

In [ ]:
repro = {
    "Model": MODEL_NAME, "Tokenizer": MODEL_NAME, "Dataset": "SQuAD v1.1 (rajpurkar/squad)",
    "Train split": f"{TRAIN_N} câu hỏi ngẫu nhiên từ train (seed={SEED})",
    "Validation split": f"{VAL_N} câu hỏi từ SQuAD validation (dùng cho learning curve)",
    "Test split": f"{TEST_N} câu hỏi còn lại từ SQuAD validation (không trùng validation)",
    "Max sequence length": f"core={MAIN_MAX_LEN}; Exp1={EXP1_LENGTHS}",
    "Stride": f"core={MAIN_STRIDE}; Exp1={EXP1_STRIDE}",
    "Batch size": BATCH_SIZE, "Learning rate": f"{LR} (linear decay, warmup {WARMUP_RATIO})",
    "Epochs": f"core={MAIN_EPOCHS}; Exp1={EXP1_EPOCHS}",
    "Optimizer": f"AdamW (weight_decay={WEIGHT_DECAY}), fp16 autocast={use_amp}, grad clip=1.0",
    "Random seed": SEED,
    "Hardware": torch.cuda.get_device_name(0) if use_amp else platform.processor() or "CPU",
    "Training time (core, s)": round(main_time, 1),
    "Decode": f"n_best={N_BEST}, max_answer_len={MAX_ANSWER_LEN} tokens",
}
display(pd.DataFrame(repro.items(), columns=["Item", "Value"]))

In [ ]:
results = []
for _, r in exp1_df.iterrows():
    results.append({"experiment": "Exp1_context_length", "setting": f"max_length={int(r['Max length'])},stride={EXP1_STRIDE},epochs={EXP1_EPOCHS}",
                    "EM": r["EM"], "F1": r["F1"], "train_time_s": r["Training time (s)"]})
results += [{"experiment": "Exp2/3_main", "setting": f"{MODEL_NAME},max_length={MAIN_MAX_LEN},stride={MAIN_STRIDE},epochs={MAIN_EPOCHS}",
             "EM": round(m_em, 2), "F1": round(m_f1, 2), "train_time_s": round(main_time, 1)},
            {"experiment": "Exp3_baseline", "setting": "keyword-overlap", "EM": round(b_em, 2), "F1": round(b_f1, 2), "train_time_s": 0}]
for _, r in exp4.iterrows():
    results.append({"experiment": "Exp4_overlap", "setting": f"group={r['group']}", "EM": r["Transformer_EM"], "F1": r["Transformer_F1"], "train_time_s": ""})
for _, r in exp5.iterrows():
    results.append({"experiment": "Exp5_answer_length", "setting": f"len={r['Answer length']}", "EM": r["Transformer_EM"], "F1": r["Transformer_F1"], "train_time_s": ""})
pd.DataFrame(results).to_csv("results.csv", index=False)
display(pd.DataFrame(results))

In [ ]:
# (Colab) tải các file kết quả về máy
try:
    from google.colab import files
    for f in ["results.csv", "learning_curves.png", "em_f1_analysis.png", "answer_length_analysis.png",
              "error_analysis_candidates.csv", "em0_f1pos_examples.csv", "exp1_context_length.png", "exp4_overlap.png"]:
        if os.path.exists(f): files.download(f)
except ImportError:
    print("Không chạy trên Colab — các file nằm trong thư mục hiện tại.")

### AI-assisted code (điền theo yêu cầu mục 29)
- **What was generated:** …
- **What was modified:** …
- **What was verified:** … *(gợi ý: bạn đã chạy các test biên ở mục 3, đối chiếu F1 tính tay với code ở mục 4, kiểm tra decode span khớp answer gốc)*

> Theo quy định, **character/token alignment, start/end calculation, prediction, metric calculation, error analysis, reflection** phải do bạn tự làm/hiểu rõ — hãy đọc kỹ các hàm `align_span`, `f1_single`, `find_best_span` và có thể tự viết lại bằng cách của bạn.

---
# 15. Individual Learning Check (mục 33)

**1. Extractive QA khác classification như thế nào?**
Classification nhận một văn bản và dự đoán **một nhãn** từ tập nhãn cố định, thường dựa trên vector `[CLS]`. Extractive QA nhận cặp *(question, context)* và dự đoán **hai vị trí** (start, end) trên các token của context; "không gian đầu ra" phụ thuộc vào độ dài context chứ không phải tập lớp cố định, và đáp án luôn là một đoạn con của context.

**2. Tại sao QA cần start và end position?**
Đáp án là một **span liên tiếp** nên cần xác định cả điểm bắt đầu lẫn điểm kết thúc; một vị trí đơn lẻ không biểu diễn được độ dài/phạm vi. Mô hình có hai bộ phân loại (start head, end head), mỗi bộ cho một phân phối softmax trên các vị trí token; khi decode ta chọn $(i,j)$ với $i \le j$ có tổng điểm cao nhất. Hai dự đoán cũng cho phép tính loss riêng $L_{start}$, $L_{end}$ và phân tích riêng wrong-start / wrong-end.

**3. Character offset khác token index như thế nào?**
*Character offset* là vị trí ký tự trong chuỗi gốc (dataset cung cấp `answer_start` theo kiểu này). *Token index* là vị trí trong chuỗi token sau tokenization (gồm cả `[CLS]`, `[SEP]`, token của question, và sub-word). Một từ có thể thành nhiều token, một token có thể phủ nhiều ký tự, và token index của context bị dịch đi bởi phần question — nên hai loại chỉ số không thể dùng thay nhau. *Word index* là vị trí theo từ (tách theo khoảng trắng), cũng khác với hai loại trên.

**4. Tại sao cần offset mapping?**
Vì nhãn của dataset ở mức ký tự còn model làm việc ở mức token. `offset_mapping` cho biết mỗi token ứng với đoạn ký tự `(start_char, end_char)` nào, nhờ đó (a) đổi `answer_start/answer_end` thành `start_token/end_token` khi tạo nhãn huấn luyện, và (b) đổi token span dự đoán ngược về chuỗi văn bản trong context khi suy luận.

**5. `stride` dùng để làm gì?**
`stride` là số token **chồng lấn** giữa hai window liên tiếp khi cắt context dài. Nếu đáp án nằm sát ranh giới của window này thì nó vẫn xuất hiện trọn vẹn (kèm ngữ cảnh xung quanh) trong window kế tiếp, tránh việc đáp án bị cắt đôi.

**6. Vì sao context dài có thể phải chia thành nhiều window?**
Transformer có độ dài chuỗi tối đa (ví dụ 512 token, và chi phí self-attention tăng theo bình phương độ dài), nên context dài hơn `max_length` không thể đưa vào một lần. Nếu chỉ truncate, đáp án nằm ở phần bị cắt sẽ mất hoàn toàn; chia thành các window chồng lấn và lấy span có điểm cao nhất trên tất cả window giúp giữ được đáp án.

**7. Exact Match đo điều gì?**
EM đo tỉ lệ câu hỏi mà chuỗi dự đoán **trùng hoàn toàn** với ít nhất một gold answer sau normalization (lowercase, bỏ dấu câu, bỏ mạo từ a/an/the, chuẩn hoá khoảng trắng). Đây là thước đo rất nghiêm ngặt: lệch một từ là bằng 0.

**8. F1 của QA khác gì với accuracy?**
Accuracy (và EM) là đúng/sai nhị phân cho từng mẫu. F1 của QA là độ chồng lấn ở mức **token** giữa prediction và gold: $P=\frac{\#overlap}{\#pred}$, $R=\frac{\#overlap}{\#gold}$, $F1=\frac{2PR}{P+R}$, tính cho từng câu rồi lấy trung bình (max theo các gold). Nó cho điểm từng phần với câu trả lời đúng một phần, khác với F1 trong phân loại vốn tính precision/recall trên các lớp.

**9. Vì sao prediction có thể có F1 > 0 nhưng EM = 0?**
Khi prediction chỉ đúng một phần: giao nhau về token nhưng không trùng hoàn toàn. Ví dụ gold = "Paris", prediction = "the city of Paris" → sau normalization là "city of paris" ≠ "paris" nên EM = 0, nhưng có 1 token chung: P = 1/3, R = 1, F1 = 0.5. Tương tự gold "Paris France" vs prediction "France": P = 1, R = 0.5, F1 ≈ 0.67.

**10. Wrong-start và wrong-end khác nhau như thế nào?**
*Wrong-start*: vị trí kết thúc đúng nhưng vị trí bắt đầu sai (ví dụ gold "New York City", pred "York City"). *Wrong-end*: vị trí bắt đầu đúng nhưng kết thúc sai (pred "New York"). Vì start head và end head là hai bộ phân loại độc lập nên chúng có thể sai độc lập; thống kê `#wrong start` vs `#wrong end` (mục 12) cho biết ranh giới nào khó hơn. *(Giả thuyết nguyên nhân: phần bạn tự rút ra từ số liệu của mình.)*

**11. Extractive QA khác RAG ở đâu?**
Extractive QA nhận *(question, context đã biết)* và trả về một **span trích nguyên văn** từ context đó. RAG gồm thêm bước **retrieval** để tìm các tài liệu liên quan từ một kho lớn (Question → Retriever → Top-K documents → Context), rồi dùng generator/QA model để trả lời; generator còn có thể tổng hợp, diễn đạt lại câu trả lời thay vì chỉ trích span.

**12. Khi nào extractive QA không thể trả lời câu hỏi?**
(a) Đáp án **không có** trong context (cần no-answer như SQuAD 2.0; nếu không, model vẫn buộc phải chọn một span); (b) đáp án đòi hỏi suy luận/tổng hợp/đếm/so sánh hoặc diễn đạt lại, không tồn tại như một đoạn liên tiếp; (c) đáp án nằm rải rác ở nhiều chỗ hoặc nhiều tài liệu; (d) phần chứa đáp án bị truncate / nằm ngoài window; (e) context không được biết trước (cần retrieval — LAB 11).

---
### ✍️ Reflection (300–500 từ — tự viết, nộp trong `reflection.md`)
*LAB 08, 09, 10 đều dùng pretrained Transformer nhưng giải quyết ba bài toán khác nhau như thế nào? Vì sao cùng backbone nhưng prediction head và target khác nhau?* — kèm sơ đồ: Pretrained Transformer → `[CLS]` rep → Classification (LAB 08); token reps → NER (LAB 09); token reps → Start/End (LAB 10).